# Fuzzy Set Operators: Visualization and Interpretation

Fuzzy set operators combine or transform membership values in the interval \([0,1]\).

In this notebook, we will visualize three important families of operators:

1. **Complement operators** — fuzzy versions of NOT  
2. **t-norms** — fuzzy versions of AND  
3. **t-conorms** — fuzzy versions of OR  

The goal is not only to produce plots, but also to **read the plots and connect their shapes to the formulas**.


## Learning objectives

By the end of this notebook, you should be able to:

- explain what a fuzzy complement does;
- compare the standard, Sugeno, and Yager complements;
- identify common t-norm and t-conorm operators;
- interpret a binary fuzzy operator as a surface over \([0,1]\times[0,1]\);
- compare operators numerically at selected membership values;
- recognize the relationship between fuzzy AND/OR operators and the boundary values 0 and 1.


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

# A consistent grid of membership values used throughout the notebook
membership = np.linspace(0.0, 1.0, 301)


## 1. Complement operators

A complement converts a membership value \(a\) into the membership of **NOT \(A\)**.

The standard fuzzy complement is

$C(a)=1-a$.

Two parameterized alternatives are the **Sugeno complement**

$C_\lambda(a)=\frac{1-a}{1+\lambda a} \lambda>-1$,

and the **Yager complement**

$C_w(a) = \left( 1-a^w \right)^{1/w}, w>0$.

All of these map values from \([0,1]\) back into \([0,1]\), but their shapes differ.

### Before running the next cell

Predict what every valid complement should do at the two endpoints:

- If \(a=0\), what should \(C(a)\) be?
- If \(a=1\), what should \(C(a)\) be?


In [ ]:
def standard_complement(a):
    return 1.0 - a

def sugeno_complement(a, lam):
    if lam <= -1:
        raise ValueError("Sugeno parameter lambda must be greater than -1.")
    return (1.0 - a) / (1.0 + lam * a)

def yager_complement(a, w):
    if w <= 0:
        raise ValueError("Yager parameter w must be greater than 0.")
    return (1.0 - a**w)**(1.0 / w)


x = membership

plt.figure(figsize=(8, 6))
plt.plot(x, standard_complement(x), label="Standard")
plt.plot(x, sugeno_complement(x, 5.0), label="Sugeno λ = 5.0")
plt.plot(x, sugeno_complement(x, -0.9), label="Sugeno λ = -0.9")
plt.plot(x, yager_complement(x, 2.0), label="Yager w = 2.0")
plt.plot(x, yager_complement(x, 0.5), label="Yager w = 0.5")

plt.xlabel("Original membership, a")
plt.ylabel("Complement membership, C(a)")
plt.title("Comparison of fuzzy complement operators")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


### Checkpoint: interpreting the complement plot

Look at the curves before answering.

1. Which complement is a straight line?
2. For values near \(a=0.5\), do all complements return the same value?
3. What happens to the Sugeno curve when \(\lambda\) changes?
4. What happens to the Yager curve when \(w\) changes?

The parameters do **not** change the endpoint behavior. They change how aggressively intermediate membership values are complemented.


In [ ]:
# Compare the operators at one membership value.
a = 0.4

print(f"a = {a}")
print(f"Standard complement       = {standard_complement(a):.4f}")
print(f"Sugeno complement λ=5.0  = {sugeno_complement(a, 5.0):.4f}")
print(f"Sugeno complement λ=-0.9 = {sugeno_complement(a, -0.9):.4f}")
print(f"Yager complement w=2.0   = {yager_complement(a, 2.0):.4f}")
print(f"Yager complement w=0.5   = {yager_complement(a, 0.5):.4f}")


## 2. t-norm operators: fuzzy AND

A **t-norm** combines two membership values \(a\) and \(b\) and plays the role of fuzzy AND.

Two common examples are:

### Minimum t-norm

$T_{\min}(a,b)=\min(a,b)$

### Product t-norm

$T_{\text{prod}}(a,b)=ab$

For example, if

$a=0.7, b=0.4$,

then the minimum t-norm gives \(0.4\), while the product t-norm gives \(0.28\).

### Prediction

Before running the visualization, think about these boundary cases:

- What should happen when one input is 0?
- What should happen when one input is 1?
- Which operator do you expect to give the larger result for most values inside the unit square?


In [ ]:
def tnorm_min(a, b):
    return np.minimum(a, b)

def tnorm_product(a, b):
    return np.multiply(a, b)


grid = np.linspace(0.0, 1.0, 41)
A, B = np.meshgrid(grid, grid)

Z_product = tnorm_product(A, B)
Z_min = tnorm_min(A, B)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

ax.plot_wireframe(A, B, Z_product, rstride=2, cstride=2, alpha=0.65, label="Product: a·b")
ax.plot_wireframe(A, B, Z_min, rstride=2, cstride=2, alpha=0.65, label="Minimum: min(a,b)")

ax.set_xlabel("a")
ax.set_ylabel("b")
ax.set_zlabel("T(a,b)")
ax.set_title("Two t-norms: fuzzy AND")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_zlim(0, 1)
ax.legend()
plt.show()


### Reading a binary-operator surface

The horizontal axes represent the two input memberships, \(a\) and \(b\).

The vertical axis represents the result:

$z=T(a,b)$.

So every point on the surface answers a question of the form:

> If the first proposition has membership \(a\) and the second has membership \(b\), what is their fuzzy AND?

A 3-D plot is useful for seeing the full operator, but numerical examples are often easier for checking your understanding.


In [ ]:
examples = [
    (0.2, 0.8),
    (0.5, 0.5),
    (0.7, 0.4),
    (1.0, 0.6),
    (0.0, 0.9),
]

print("   a     b    min(a,b)    a*b")
print("--------------------------------")
for a, b in examples:
    print(f"{a:4.1f}  {b:4.1f}     {tnorm_min(a,b):.3f}      {tnorm_product(a,b):.3f}")


## 3. t-conorm operators: fuzzy OR

A **t-conorm** combines two membership values and plays the role of fuzzy OR.

Two common examples are:

### Maximum t-conorm

$S_{\max}(a,b)=\max(a,b)$

### Probabilistic sum

$S_{\text{prob}}(a,b)=a+b-ab$

The probabilistic sum can also be written as

$1-(1-a)(1-b)$.

### Prediction

Before running the next cell:

- What happens if one input equals 0?
- What happens if one input equals 1?
- Which of these two OR operators do you expect to produce the larger value inside the unit square?


In [ ]:
def tconorm_max(a, b):
    return np.maximum(a, b)

def tconorm_prob_sum(a, b):
    return a + b - a*b


Z_prob = tconorm_prob_sum(A, B)
Z_max = tconorm_max(A, B)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

ax.plot_wireframe(A, B, Z_prob, rstride=2, cstride=2, alpha=0.65,
                  label="Probabilistic sum: a+b-ab")
ax.plot_wireframe(A, B, Z_max, rstride=2, cstride=2, alpha=0.65,
                  label="Maximum: max(a,b)")

ax.set_xlabel("a")
ax.set_ylabel("b")
ax.set_zlabel("S(a,b)")
ax.set_title("Two t-conorms: fuzzy OR")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_zlim(0, 1)
ax.legend()
plt.show()


In [ ]:
print("   a     b    max(a,b)   a+b-a*b")
print("----------------------------------")
for a, b in examples:
    print(f"{a:4.1f}  {b:4.1f}     {tconorm_max(a,b):.3f}      {tconorm_prob_sum(a,b):.3f}")


## 4. A 2-D view can be easier to read

A 3-D surface is visually appealing, but a **filled contour plot** often makes it easier to compare numerical regions.

Below, the same probabilistic-sum t-conorm is displayed from directly above. Each contour corresponds to a similar output value.


In [ ]:
plt.figure(figsize=(7, 6))
contours = plt.contourf(A, B, Z_prob, levels=20)
plt.colorbar(contours, label="S(a,b) = a + b - ab")

plt.xlabel("a")
plt.ylabel("b")
plt.title("Probabilistic-sum t-conorm: contour view")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.show()


### Why this is preferable to interpolation from only four corner points

The original version of this notebook demonstrated `matplotlib.tri.CubicTriInterpolator` using the four corners of the unit square.

That technique can be useful when we only have **sampled data** and want to estimate values between samples. Here, however, we already know the fuzzy operator formula exactly:

\[
S(a,b)=a+b-ab.
\]

Therefore, evaluating the formula directly on a dense grid is clearer and avoids introducing interpolation error into a function we already know.

Interpolation becomes more relevant when the surface is known only from measured or sampled points.


## 5. AND versus OR at the same input

Consider

\[
a=0.7,\qquad b=0.4.
\]

Before running the next cell, rank the following four results from smallest to largest:

- product AND;
- minimum AND;
- maximum OR;
- probabilistic-sum OR.


In [ ]:
a, b = 0.7, 0.4

results = {
    "Product AND": tnorm_product(a, b),
    "Minimum AND": tnorm_min(a, b),
    "Maximum OR": tconorm_max(a, b),
    "Probabilistic-sum OR": tconorm_prob_sum(a, b),
}

for name, value in sorted(results.items(), key=lambda item: item[1]):
    print(f"{name:22s} -> {value:.3f}")


## 6. Quick property checks

The operators above satisfy several useful boundary relationships.

For a t-norm \(T\):

\[
T(a,0)=0,
\qquad
T(a,1)=a.
\]

For a t-conorm \(S\):

\[
S(a,0)=a,
\qquad
S(a,1)=1.
\]

These properties are useful sanity checks when implementing fuzzy operators in code.


In [ ]:
test_values = np.array([0.0, 0.2, 0.5, 0.8, 1.0])

print("Minimum t-norm:")
print("T(a, 0) =", tnorm_min(test_values, 0))
print("T(a, 1) =", tnorm_min(test_values, 1))

print("\nProduct t-norm:")
print("T(a, 0) =", tnorm_product(test_values, 0))
print("T(a, 1) =", tnorm_product(test_values, 1))

print("\nMaximum t-conorm:")
print("S(a, 0) =", tconorm_max(test_values, 0))
print("S(a, 1) =", tconorm_max(test_values, 1))

print("\nProbabilistic-sum t-conorm:")
print("S(a, 0) =", tconorm_prob_sum(test_values, 0))
print("S(a, 1) =", tconorm_prob_sum(test_values, 1))


# Practice

Try these without immediately copying the earlier cells.

### Exercise 1 — complements

For \(a=0.65\), compute:

1. the standard complement;
2. the Sugeno complement with \(\lambda=2\);
3. the Yager complement with \(w=2\).

First calculate them by hand, then verify with Python.

### Exercise 2 — fuzzy AND

Let \(a=0.35\) and \(b=0.80\).

Compute both:

$T_{\min}(a,b)$

and

$T_{\text{prod}}(a,b)$.

Which one is larger?

### Exercise 3 — fuzzy OR

Using the same values, compute:

$S_{\max}(a,b)$

and

$S_{\text{prob}}(a,b)$.

Which one is larger?

### Exercise 4 — visualization

Create a contour plot for the **product t-norm**

$T(a,b)=ab$.

Use the existing `A` and `B` grids.

### Challenge

Choose several values of \(a\) and \(b\) and investigate whether the following inequalities appear to hold:

$ab \leq \min(a,b)$

and

$\max(a,b) \leq a+b-ab$.

Can you explain why?


In [ ]:
# Exercise workspace
# Add your code below.


## Summary

We visualized several common fuzzy operators.

- **Complements** transform one membership value and represent fuzzy NOT.
- **t-norms** combine two memberships and represent fuzzy AND.
- **t-conorms** combine two memberships and represent fuzzy OR.
- Different operators can satisfy the same general role while producing different intermediate values.
- 3-D surfaces show the full binary operator, while numerical examples and contour plots can make interpretation easier.

When choosing an operator in a fuzzy system, the formula matters because it determines how strongly intermediate membership values interact.
